In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

csv_path = (
    ROOT / "data" / "raw"
    / "四川省川剧院_时长及互动数据.csv"
)

df = pd.read_csv(csv_path, encoding="utf-8-sig")
display(df.head())

,序号,作品标题,类型,时长（分:秒）,时长（秒）,作品ID,来源链接,点赞数,分享数,评论数,互动采集日期,计数说明
0,1,百戏聚昆山，省川赴新程。 由四川省川剧院二级演员杨坤昊主演的《肖方杀船》将于9月24日 19...,视频,01:14,74.0,7688661826673839411,https://www.douyin.com/video/7688661826673839411,176,4.0,12,2026-09-24,NaN
1,2,《肖方杀船》中最负盛名的看点便是“藏刀”——将钢刀藏于身上，时隐时现，防不胜防。 刀不见影，...,视频,00:35,35.0,7688291532352392474,https://www.douyin.com/video/7688291532352392474,379,35.0,32,2026-09-24,NaN
2,3,省川《肖方杀船》即将亮相2026年戏曲百戏（昆山）盛典经典折子戏（绝技绝活）展演！ 汇中国百...,视频,01:14,74.0,7687895826277879081,https://www.douyin.com/video/7687895826277879081,59,NaN,3,2026-09-24,分享未显示数字
3,4,“川味儿老苏”和“吴越东坡”成都梦幻联动，川、越共品东坡千古风骨！ 9月10日，四川省川剧院...,视频,01:22,82.0,7684227973855087918,https://www.douyin.com/video/7684227973855087918,175,18.0,8,2026-09-24,NaN
4,5,“花果山前旌旗动，水帘洞中藏劲风。七十二变不相同，四方独尊孙悟空……” 00后文武小生梁浩龙...,视频,01:32,92.0,7683462866351017267,https://www.douyin.com/video/7683462866351017267,49,1.0,6,2026-09-24,NaN


In [5]:
import re

# Create a separate working copy with clearer column names.
clean = df.rename(columns={
    "点赞数": "likes",
    "评论数": "comments",
    "分享数": "shares",
    "时长（秒）": "duration_seconds"
}).copy()


# Convert one interaction count into a number.
def parse_count(value):
    if pd.isna(value):
        return float("nan")

    text = str(value).strip().replace(",", "")

    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)\s*([万亿]?)",
        text
    )

    if not match:
        return float("nan")

    multiplier = {"": 1, "万": 10_000, "亿": 100_000_000}

    return float(match.group(1)) * multiplier[match.group(2)]


# Apply the function to each interaction column.
count_columns = ["likes", "comments", "shares"]

for column in count_columns:
    original = clean[column].copy()
    clean[column] = original.map(parse_count)

    # Find values that were present but could not be converted.
    failures = original.notna() & clean[column].isna()

    print(f"{column}: {failures.sum()} unparsed values")
    if failures.any():
        display(original[failures].value_counts(dropna=False))


# Display the results.
display(clean[count_columns].head(10))
display(clean[count_columns].dtypes)

likes: 0 unparsed values
comments: 0 unparsed values
shares: 0 unparsed values


,likes,comments,shares
0,176.0,12.0,4.0
1,379.0,32.0,35.0
2,59.0,3.0,NaN
3,175.0,8.0,18.0
4,49.0,6.0,1.0
5,150.0,16.0,21.0
6,107.0,7.0,4.0
7,28.0,1.0,3.0
8,51.0,1.0,2.0
9,38.0,1.0,3.0


likes       float64
comments    float64
shares      float64
dtype: object

In [8]:
output_folder = ROOT / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

output_path = output_folder / "sichuan_opera_cleaned.csv"
clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Save to:", output_path)

Save to: /Users/tanweiyi/Documents/Projects/sichuan-opera-engagement/data/processed/sichuan_opera_cleaned.csv
